# LLMSTU full run — bucket zip → shard-by-shard → Google Drive
Avoids HF rate-limits: downloads the **single zip** from your Storage Bucket once,
then processes shard-by-shard from local disk (crop → dedup → caption) and saves each
finished shard as **one zip to Google Drive**. No HF uploads during GPU time — you
upload the zips to HF later (off-GPU). Then build the golden set + finalize.

Runs Qwen3.5 via **transformers + sdpa** (vLLM needs CUDA 13; Colab is 12).
Use your **~95GB GPU** runtime.

## 1. Setup

In [ ]:
REPO = "vaelkokach/LLMSTU-pipeline"
import os
try:
    from google.colab import userdata
    def _secret(n):
        try: return userdata.get(n)
        except Exception: return None
except Exception:
    def _secret(n): return None
if REPO and not os.path.isdir("/content/LLMSTU-pipeline"):
    gh=_secret("GITHUB_TOKEN"); auth=f"{gh}@" if gh else ""
    !git clone https://{auth}github.com/{REPO}.git /content/LLMSTU-pipeline
if os.path.isdir("/content/LLMSTU-pipeline"): %cd /content/LLMSTU-pipeline
!pip install -q "ultralytics>=8.3.0" "transformers>=4.57.0" accelerate hf_xet \
    "huggingface_hub>=0.35.0" pandas pyarrow pyyaml pillow numpy
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"     # keep the tab responsive
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_TOKEN"] = _secret("HF_TOKEN") or ""
assert os.environ["HF_TOKEN"], "Set HF_TOKEN in Colab Secrets"
from huggingface_hub import login; login(os.environ["HF_TOKEN"])
import torch; print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Mount Google Drive\nShard zips are written here, so they persist across disconnects and sync to your PC.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
OUT_DIR = "/content/drive/MyDrive/LLMSTU_out"   # shard zips land here
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("shard zips ->", OUT_DIR)

## 3. Config check

In [ ]:
from llmstu.config import load
cfg = load("config.yaml")
print("model:", cfg.caption.model_id, "| backend:", cfg.caption.backend,
      "| prompt:", cfg.caption.prompt_name, "| schema:", cfg.caption.schema_name)
print("crop use_pose:", cfg.crop.use_pose, "| down_extend:", cfg.crop.down_extend,
      "| dedup thr:", cfg.dedup.hamming_threshold)

## 4. Find the zip inside your bucket
Copy the zip's path from the list into `ZIP` below. The zip can be a **single flat folder of images** (no per-shard subfolders needed) — the run shards it automatically into groups of 5,000.

In [ ]:
!python scripts/16_run_local.py --bucket CHANGE_ME/your-frames-dataset --list

## 5. Run — download zip once, process shards, save zips to Drive
Set `ZIP` to the path from step 4. Re-run this cell after any disconnect: shards whose
zip already exists in Drive are skipped. GPU time = crop + caption only (no HF upload).

In [ ]:
ZIP = "archive.zip"   # your bucket zip (36.8GB, single flat folder inside)
!df -h /content | tail -1   # need ~120GB+ free: 37GB zip + 37GB extract + ~55GB model
!HF_HUB_DISABLE_PROGRESS_BARS=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
    python scripts/16_run_local.py --bucket CHANGE_ME/your-frames-dataset --zip {ZIP} \
    --repo CHANGE_ME/your-crops-dataset --output zip --out-dir {OUT_DIR}

## 6. Upload the shard zips to HF — run LATER / off-GPU
You can run this on a CPU runtime, or even on your own PC (point --zips-dir at your
downloaded Drive folder). Resumable; writes the _done markers 14/13 rely on.

In [ ]:
!python scripts/17_upload_zips.py --zips-dir {OUT_DIR} --repo CHANGE_ME/your-crops-dataset

## 7. Build the golden set (after zips are uploaded to HF)

In [ ]:
!python scripts/14_golden_from_hf.py --repo CHANGE_ME/your-crops-dataset --n 1500 \
    --stratify-fields engagement_level,activity,gaze_direction --min-per-class 40 --hard-frac 0.2
!cd labeling && zip -qr /content/labeling_full.zip index.html data.json
from google.colab import files; files.download("/content/labeling_full.zip")

## 8. Hand-label + score\nUnzip, label in index.html, Export golden.json, upload it back, then score.

In [ ]:
from google.colab import files
up = files.upload()   # your golden.json
import shutil; shutil.move(list(up)[0], "labeling/golden.json"); print("saved labeling/golden.json")

In [ ]:
!python scripts/07_eval_golden.py --golden labeling/golden.json
from IPython.display import HTML
HTML(open("work/eval/golden_report.html").read())

## 9. Finalize\nBuilds crops/metadata.jsonl (image+caption join) + parquet and pushes the golden set.

In [ ]:
!python scripts/13_finalize.py --repo CHANGE_ME/your-crops-dataset --golden labeling/golden.json